```{contents}
```

## Weight Updation and the Chain Rule

### Motivation

Training a neural network means **adjusting weights** so that the model’s predictions minimize a **loss function**.
This adjustment is performed using **gradient-based optimization**, where gradients are computed using the **chain rule of calculus** and applied through **weight update rules**.

---

### Core Concepts

#### Computational Graph

A neural network defines a function:

$$
\hat{y} = f(x; W)
$$

with loss:

$$
\mathcal{L} = \ell(\hat{y}, y)
$$

Training requires computing:

$$
\frac{\partial \mathcal{L}}{\partial W}
$$

This derivative is computed efficiently using **backpropagation**, which is a systematic application of the **chain rule** on the computational graph.

---

### Chain Rule in Deep Learning

For composed functions:

$$
z = f(u), \quad u = g(x)
$$

The derivative is:

$$
\frac{dz}{dx} = \frac{dz}{du} \cdot \frac{du}{dx}
$$

In neural networks:

$$
\mathcal{L}(W) = \ell(f(x; W))
$$

For layer $k$:

$$
\frac{\partial \mathcal{L}}{\partial W_k}
= \frac{\partial \mathcal{L}}{\partial a_k}
\cdot \frac{\partial a_k}{\partial z_k}
\cdot \frac{\partial z_k}{\partial W_k}
$$

where

* $z_k = W_k a_{k-1} + b_k$
* $a_k = \sigma(z_k)$

This product structure is the backbone of **backpropagation**.

---

### Weight Updation Rule

Using **Gradient Descent**:

$$
W \leftarrow W - \eta \frac{\partial \mathcal{L}}{\partial W}
$$

* $\eta$ — learning rate
* $\nabla_W \mathcal{L}$ — gradient from backpropagation

---

### End-to-End Training Workflow

1. **Forward pass**
2. **Compute loss**
3. **Backward pass (chain rule)**
4. **Update weights**
5. **Repeat**

---

### Concrete Example (Single Neuron)

$$
z = wx + b,\quad \hat{y} = \sigma(z)
$$

$$
\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2
$$

Applying chain rule:

$$
\frac{\partial \mathcal{L}}{\partial w}
= (\hat{y}-y)\cdot \sigma'(z) \cdot x
$$

This shows explicitly how each component contributes.

---

### Demonstration in PyTorch

```python
import torch
import torch.nn as nn

# Simple model
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Data
x = torch.tensor([[2.0]])
y = torch.tensor([[4.0]])

# Forward pass
y_hat = model(x)
loss = criterion(y_hat, y)

# Backward pass (chain rule happens here)
loss.backward()

# Inspect gradients
for name, param in model.named_parameters():
    print(name, param.grad)

# Weight update
optimizer.step()
optimizer.zero_grad()
```

PyTorch builds a **dynamic computational graph** and applies the **chain rule automatically** via `autograd`.

---

### Visualization of Gradient Flow

| Component | Mathematical Meaning             |
| --------- | -------------------------------- |
| Loss      | Error signal                     |
| Backward  | Chain rule propagation           |
| Gradient  | Sensitivity of loss to parameter |
| Optimizer | Applies weight update            |

---

### Variants of Weight Update Algorithms

| Optimizer | Update Rule                        |
| --------- | ---------------------------------- |
| SGD       | $W \leftarrow W - \eta \nabla L$ |
| Momentum  | Adds velocity term                 |
| RMSProp   | Normalizes by running variance     |
| Adam      | Combines momentum + RMSProp        |

---

### Why Chain Rule Is Central

Without the chain rule:

* Deep networks would be computationally infeasible
* Gradients could not propagate through many layers
* Learning would not scale with depth

The chain rule transforms deep learning from a conceptual model into a **trainable system**.

---

### Summary

| Concept            | Role                              |
| ------------------ | --------------------------------- |
| Chain Rule         | Computes gradients efficiently    |
| Backpropagation    | Applies chain rule layer by layer |
| Weight Update      | Moves model toward lower loss     |
| Autograd (PyTorch) | Automates differentiation         |
| Optimizers         | Control convergence behavior      |

This mechanism is the mathematical engine of all modern deep learning systems.


### Weight Update Mechanisms in Deep Learning


**Weight updation** is the core learning process in neural networks.
Given a loss function $\mathcal{L}(\theta)$, training seeks parameters $\theta$ that minimize expected loss by iteratively adjusting weights using gradient information.

At each iteration:
$$
\theta_{t+1} = \theta_t - \eta \cdot \Delta_t
$$
where

* $\eta$ = learning rate
* $\Delta_t$ = update direction computed from gradients and optimizer state

Different **weight update methods** define how $\Delta_t$ is computed.

---

### Fundamental Categories of Weight Updates

| Category                  | Principle               | Purpose                          |
| ------------------------- | ----------------------- | -------------------------------- |
| Gradient Descent Variants | Uses raw gradients      | Core optimization                |
| Momentum-Based            | Accumulates velocity    | Faster convergence               |
| Adaptive Learning Rate    | Per-parameter step size | Handles sparse & noisy gradients |
| Second-Order Inspired     | Curvature-aware scaling | Stable optimization              |

---

### 1. Gradient Descent Family

#### Batch Gradient Descent

$$
\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}_{\text{full dataset}}
$$

Characteristics:

* Stable
* Very slow on large datasets

#### Stochastic Gradient Descent (SGD)

$$
\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}(x_i)
$$

Characteristics:

* Noisy but fast
* Helps escape local minima

#### Mini-Batch SGD (standard in practice)

$$
\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}_{\text{batch}}
$$

---

### 2. Momentum-Based Updates

#### Momentum

Adds velocity $v_t$

$$
v_t = \beta v_{t-1} + (1-\beta)\nabla \mathcal{L}
$$
$$
\theta \leftarrow \theta - \eta v_t
$$

Effect:

* Dampens oscillations
* Accelerates learning in consistent directions

#### Nesterov Accelerated Gradient (NAG)

Looks ahead before computing gradient:

$$
v_t = \beta v_{t-1} + \nabla \mathcal{L}(\theta - \eta \beta v_{t-1})
$$

---

### 3. Adaptive Learning Rate Methods

| Optimizer | Key Idea                                        |
| --------- | ----------------------------------------------- |
| AdaGrad   | Large updates for rare features                 |
| RMSProp   | Exponential moving average of squared gradients |
| Adam      | Momentum + RMSProp combined                     |
| AdamW     | Adam with proper weight decay                   |

#### Adam Update Rule

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t
$$
$$
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2
$$

Bias correction:
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}
$$

Final update:
$$
\theta \leftarrow \theta - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

---

### 4. Second-Order Inspired Methods

Approximate curvature of loss surface.

| Method  | Core Idea                              |
| ------- | -------------------------------------- |
| Newton  | Uses Hessian inverse                   |
| L-BFGS  | Memory-efficient Hessian approximation |
| Shampoo | Kronecker-factorized curvature         |

Rare in deep nets due to cost but powerful for certain models.

---

### PyTorch Demonstration

```python
import torch
import torch.nn as nn
import torch.optim as optim

# Simple model
model = nn.Sequential(
    nn.Linear(10, 1)
)

loss_fn = nn.MSELoss()

# Try different optimizers
optimizers = {
    "SGD": optim.SGD(model.parameters(), lr=0.1),
    "Momentum": optim.SGD(model.parameters(), lr=0.1, momentum=0.9),
    "Adam": optim.Adam(model.parameters(), lr=0.01),
    "AdamW": optim.AdamW(model.parameters(), lr=0.01)
}

# Dummy data
x = torch.randn(32, 10)
y = torch.randn(32, 1)

for name, optimizer in optimizers.items():
    optimizer.zero_grad()
    output = model(x)
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()
    print(f"{name} update applied, loss = {loss.item():.4f}")
```

---

### Comparative Summary

| Optimizer | Convergence Speed        | Stability | Tuning Difficulty | Common Usage       |
| --------- | ------------------------ | --------- | ----------------- | ------------------ |
| SGD       | Medium                   | Medium    | High              | Vision models      |
| Momentum  | Fast                     | High      | Medium            | CNNs               |
| Adam      | Very Fast                | Very High | Low               | NLP, Transformers  |
| AdamW     | Very Fast                | Very High | Low               | Modern deep models |
| L-BFGS    | Very Fast (small models) | High      | High              | Scientific ML      |

---

### Practical Guidelines

* **Start with AdamW** for most problems.
* Switch to **SGD + Momentum** for large vision models when fine-tuning.
* Tune learning rate before any other hyperparameter.
* Combine with learning rate schedulers for best results.

---

### Conceptual Insight

Weight update mechanisms determine:

* how efficiently a model navigates the loss surface,
* how well it escapes poor local minima,
* and how stable training remains under noisy gradients.

Choosing the optimizer is often the most impactful training decision after model architecture.